# 🚀 QuantDataPipeline — 量化數據中台

**全自動一鍵執行**：輸入參數 → 按下播放鍵 → 自動下載 + Greeks 計算 + 多週期特徵聚合

---

### 📋 使用說明
1. 在下方表單填入 **FinMind API Token**
2. 設定 **每小時 API 額度** (帳號等級對應的 requests/hour)
3. 選擇 **GitHub 分支**、**回溯天數** (填 `0` = 全量抓取到 2011-01-03)
4. 按下左側 ▶️ 播放鍵即可

> ⚠️ 首次執行會自動安裝依賴套件（約 30 秒），後續執行會直接跳過。
>
> ⚠️ 2019-01-16 ~ 2019-06-30 期間部分資料不完整 (FinMind 官方已知缺失)。

In [ ]:
#@title 🎛️ QuantDataPipeline 控制面板 { run: "auto", display-mode: "form" }
#@markdown ---
#@markdown ### 🔑 API 設定
FINMIND_API_TOKEN = '' #@param {type:"string"}
API_QUOTA_PER_HOUR = 1600 #@param {type:"integer"}
#@markdown ---
#@markdown ### 🌿 GitHub 分支
BRANCH = '3' #@param {type:"string"}
#@markdown ---
#@markdown ### ⚙️ 管線參數
LOOKBACK_DAYS = 30 #@param {type:"integer"}
#@markdown > `0` = 全量抓取 (從 2011-01-03 至今)。資料區間: 2011-01-03 ~ now
SKIP_GREEKS = False #@param {type:"boolean"}
#@markdown ---
#@markdown ### 💾 儲存設定
SYNC_TO_DRIVE = True #@param {type:"boolean"}
DRIVE_PATH = '/content/drive/MyDrive/QuantData' #@param {type:"string"}
DRIVE_SYNC_EVERY = 10 #@param {type:"integer"}
#@markdown > 每處理 N 天後自動同步一次到 Drive (防止 Colab 斷線遺失資料)
#@markdown ---

# ═══════════════════════════════════════════════════════════════
# 以下為自動執行邏輯，不需修改
# ═══════════════════════════════════════════════════════════════

import subprocess, sys, os, time, shutil, json, logging, math, importlib
from datetime import datetime, timedelta
from pathlib import Path
from IPython.display import display, HTML

# ── 清除舊模組快取 ──
stale_prefixes = ['core.', 'fetchers.', 'processors.', 'storage.', 'compute_greeks']
for mod_name in list(sys.modules.keys()):
    if any(mod_name.startswith(p) or mod_name == p.rstrip('.') for p in stale_prefixes):
        del sys.modules[mod_name]

os.environ['FINMIND_API_TOKEN'] = FINMIND_API_TOKEN
os.environ['FINMIND_QUOTA_PER_HOUR'] = str(API_QUOTA_PER_HOUR)

for name in ['pipeline', 'pipeline.retry', 'pipeline.extractor',
             'pipeline.orchestrator', 'pipeline.http', 'pipeline.rate_limiter',
             'pipeline.schema']:
    logging.getLogger(name).setLevel(logging.CRITICAL)

display(HTML("""
<style>
  .output_scroll { height: 420px !important; overflow-y: auto !important; }
  .output_wrapper { max-height: 420px !important; overflow-y: auto !important; }
  .output_area pre { font-family: 'Fira Code', 'Consolas', monospace; font-size: 13px; line-height: 1.6; }
</style>
<script>
  (function() {
    var output = document.querySelector('.output_scroll, .output_wrapper');
    if (output) { output.style.maxHeight = '420px'; output.style.overflowY = 'auto'; }
    var observer = new MutationObserver(function() {
      var el = document.querySelector('.output_scroll, .output_wrapper');
      if (el) el.scrollTop = el.scrollHeight;
    });
    var target = document.querySelector('.output_area');
    if (target) observer.observe(target, {childList: true, subtree: true});
  })();
</script>
"""))

api_call_count = 0
pipeline_start_time = time.time()

def log(icon, msg):
    ts = datetime.now().strftime('%H:%M:%S')
    print(f"{icon} {ts} | {msg}", flush=True)

def header(title):
    print(f"\n{'━'*55}", flush=True)
    print(f"  {title}", flush=True)
    print(f"{'━'*55}", flush=True)

FATAL_KEYWORDS = ['user level', 'update your user level', 'please upgrade',
                  'permission denied', 'invalid token', 'unauthorized', 'sponsor']
def is_fatal(e):
    return any(kw in str(e).lower() for kw in FATAL_KEYWORDS)

# ── 日期計算 ──
DATA_EARLIEST = '2011-01-03'
today = datetime.now()
end_date = today.strftime('%Y-%m-%d')
if LOOKBACK_DAYS <= 0:
    start_date = DATA_EARLIEST
    lookback_label = f'全量 ({DATA_EARLIEST} ~ {end_date})'
else:
    start_date = (today - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
    lookback_label = f'{LOOKBACK_DAYS} 天 ({start_date} ~ {end_date})'

CALLS_PER_DAY = 2
rate_delay = round(3600 / max(API_QUOTA_PER_HOUR, 1) * 1.1, 2)
total_calendar_days = (today - datetime.strptime(start_date, '%Y-%m-%d')).days
est_trading_days = int(total_calendar_days * 0.66)
est_api_calls = est_trading_days * CALLS_PER_DAY + 1
est_minutes = round(est_api_calls * rate_delay / 60, 1)
est_hours = round(est_minutes / 60, 1)

header('🚀 QuantDataPipeline 啟動中')
log('📅', f'範圍: {lookback_label}')
log('🔧', f'Greeks: {"開" if not SKIP_GREEKS else "關"} | Drive: {"開" if SYNC_TO_DRIVE else "關"} | 分支: {BRANCH}')
log('💾', f'每 {DRIVE_SYNC_EVERY} 天自動同步到 Drive')
print(flush=True)
log('📊', f'═══ API 用量預估 ═══')
log('  ', f'額度:      {API_QUOTA_PER_HOUR} 次/hr')
log('  ', f'速率:      {rate_delay}s / 請求')
log('  ', f'預估交易日: ~{est_trading_days} 天')
log('  ', f'預估呼叫數: ~{est_api_calls} 次')
if est_hours >= 1:
    log('  ', f'預估耗時:  ~{est_hours} 小時')
else:
    log('  ', f'預估耗時:  ~{est_minutes} 分鐘')

# ═══════════════════════════════════════════════════════════════
# Phase 0: 環境準備
# ═══════════════════════════════════════════════════════════════
header('📦 Phase 0: 環境準備')

REPO_URL = 'https://github.com/hsp1234-web/SP_OP_20260220.git'
REPO_DIR = Path('/content/SP_OP_20260220')

if SYNC_TO_DRIVE:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        log('✅', 'Google Drive 已掛載')
    except Exception as e:
        log('⚠️', f'Drive 掛載失敗: {e}')
        SYNC_TO_DRIVE = False

if REPO_DIR.exists():
    log('🔄', f'更新代碼 (分支 {BRANCH})...')
    subprocess.run(['git', 'fetch', 'origin'], cwd=str(REPO_DIR), capture_output=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=str(REPO_DIR), capture_output=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], cwd=str(REPO_DIR), capture_output=True)
else:
    log('📥', f'下載代碼 (分支 {BRANCH})...')
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], capture_output=True)

PROJECT_DIR = REPO_DIR / 'QuantDataPipeline'
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))

log('📦', '安裝依賴套件...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'polars', 'numba', 'scipy', 'numpy', 'requests', 'python-dotenv'],
    capture_output=True, text=True
)
log('✅', '依賴套件就緒')

env_path = PROJECT_DIR / '.env'
env_path.write_text(f'FINMIND_API_TOKEN={FINMIND_API_TOKEN}\nFINMIND_QUOTA_PER_HOUR={API_QUOTA_PER_HOUR}\n')
log('🔑', f'API Token: {"已設定" if FINMIND_API_TOKEN else "未設定 (匿名模式)"}')

# ═══════════════════════════════════════════════════════════════
# 初始化管線
# ═══════════════════════════════════════════════════════════════
os.chdir(str(PROJECT_DIR))

from core.config import DATA_DIR, DB_PATH, RATE_LIMIT_DELAY
from core.db_metadata_manager import DBManager
from core.fetch_orchestrator import process_task, seed_tasks_from_dates
from fetchers.datasets.technical import trading_date
from fetchers.infrastructure.http_session import get_session

drive_data = Path(DRIVE_PATH) if SYNC_TO_DRIVE else None

if SYNC_TO_DRIVE and drive_data:
    drive_db = drive_data / 'status.db'
    if drive_db.exists():
        shutil.copy2(drive_db, DB_PATH)
        log('📋', '已從 Drive 還原 status.db')

DBManager._reset_instance()
db = DBManager(DB_PATH)
session = get_session()

# ── Drive 同步函數 ──
def sync_to_drive():
    if not SYNC_TO_DRIVE or not drive_data:
        return 0
    drive_data.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DB_PATH, drive_data / 'status.db')
    synced = 0
    if DATA_DIR.exists():
        for pq in DATA_DIR.rglob('*.parquet'):
            rel = pq.relative_to(DATA_DIR)
            dest = drive_data / 'data' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists() or dest.stat().st_size != pq.stat().st_size:
                shutil.copy2(pq, dest)
                synced += 1
    return synced

# ═══════════════════════════════════════════════════════════════
# 逐日處理：下載 → 計算 → 定期同步
# ═══════════════════════════════════════════════════════════════
header('⚡ 逐日處理 (下載 → Greeks → 同步)')
log('⏱️', f'實際速率: {RATE_LIMIT_DELAY}s / 請求')
log('📅', f'日期範圍: {start_date} → {end_date}')

# 取得交易日清單
trading_dates_list = []
try:
    trading_dates_df = trading_date.fetch_trading_dates(session, start_date, end_date)
    api_call_count += 1
    if trading_dates_df is not None and not trading_dates_df.is_empty():
        date_col = 'date' if 'date' in trading_dates_df.columns else trading_dates_df.columns[0]
        trading_dates_list = sorted(trading_dates_df[date_col].cast(str).to_list())
        # 只取 YYYY-MM-DD 部分
        trading_dates_list = [d[:10] for d in trading_dates_list]
        log('✅', f'找到 {len(trading_dates_list)} 個交易日')
    else:
        log('⚠️', '此範圍內無交易日')
except Exception as e:
    if is_fatal(e):
        log('⛔', f'API 權限不足: {str(e)[:60]}')
    else:
        log('❌', f'取得交易日失敗: {e}')

if not SKIP_GREEKS:
    from compute_greeks_pipeline import compute_greeks_for_date

pipeline_aborted = False
COOLDOWN = 300
total_days = len(trading_dates_list)
days_done = 0
days_downloaded = 0
days_greeks = 0
days_skipped = 0
total_synced = 0

log('📊', f'開始逐日處理 {total_days} 個交易日...')
print(flush=True)

for day_idx, tdate in enumerate(trading_dates_list, 1):
    if pipeline_aborted:
        break

    year = tdate.split('-')[0]
    day_label = f'[{day_idx}/{total_days}]'

    # ── Step 1: 下載 TXO + TX ──
    txo_ok = False
    tx_ok = False

    for dset, did in [('TaiwanOptionTick', 'TXO'), ('TaiwanFuturesTick', 'TX')]:
        task_id = f'{tdate}_{dset}_{did}'
        existing = db.get_task_status(task_id)
        if existing and existing >= 1:
            # 已下載過，跳過
            if dset == 'TaiwanOptionTick':
                txo_ok = True
            else:
                tx_ok = True
            continue

        try:
            db.register_task(task_id, tdate, dset, did)
            process_task(task_id, tdate, dset, did)
            api_call_count += 1
            status = db.get_task_status(task_id)
            if status == 1:
                if dset == 'TaiwanOptionTick':
                    txo_ok = True
                else:
                    tx_ok = True
            elif status == 3:
                pass  # 無資料
        except Exception as e:
            api_call_count += 1
            if is_fatal(e):
                log('⛔', f'帳號權限不足，管線中止')
                log('💡', f'請設定 FINMIND_API_TOKEN (需 backer/sponsor 等級)')
                pipeline_aborted = True
                break
            err = str(e).lower()
            if any(k in err for k in ['429', 'rate limit', 'quota', 'too many']):
                log('🧊', f'API 限速 — 冷卻 {COOLDOWN}s...')
                time.sleep(COOLDOWN)
                try:
                    process_task(task_id, tdate, dset, did)
                    api_call_count += 1
                    status = db.get_task_status(task_id)
                    if status == 1:
                        if dset == 'TaiwanOptionTick':
                            txo_ok = True
                        else:
                            tx_ok = True
                except:
                    pass

    if pipeline_aborted:
        break

    # ── Step 2: 計算 Greeks (若 TXO + TX 都成功) ──
    greeks_done = False
    if txo_ok and tx_ok and not SKIP_GREEKS:
        gpath = DATA_DIR / year / 'GreeksFeatures' / f'TXO_Greeks_{tdate}.parquet'
        if gpath.exists():
            greeks_done = True
        else:
            try:
                df, opath, ok = compute_greeks_for_date(tdate)
                if ok:
                    db.update_task_status(f'{tdate}_TaiwanOptionTick_TXO', 2)
                    greeks_done = True
                    days_greeks += 1
            except Exception as e:
                log('❌', f'{tdate} | Greeks 失敗: {str(e)[:40]}')

    # ── 輸出狀態 ──
    if not txo_ok and not tx_ok:
        st = db.get_task_status(f'{tdate}_TaiwanOptionTick_TXO')
        if st == 3:
            log('➖', f'{tdate} | 無資料跳過 {day_label}')
            days_skipped += 1
        else:
            log('❌', f'{tdate} | 下載失敗 {day_label}')
    elif greeks_done:
        log('✅', f'{tdate} | 下載+Greeks 完成 {day_label}')
        days_downloaded += 1
    elif txo_ok or tx_ok:
        if SKIP_GREEKS:
            log('✅', f'{tdate} | 下載完成 {day_label}')
        else:
            log('⚠️', f'{tdate} | 下載完成，Greeks 待處理 {day_label}')
        days_downloaded += 1

    days_done += 1

    # ── Step 3: 定期同步到 Drive ──
    if SYNC_TO_DRIVE and days_done % DRIVE_SYNC_EVERY == 0:
        n = sync_to_drive()
        total_synced += n
        log('☁️', f'Drive 同步: {n} 個檔案 (累計 {total_synced}) {day_label}')

# ═══════════════════════════════════════════════════════════════
# 最終同步
# ═══════════════════════════════════════════════════════════════
if SYNC_TO_DRIVE and not pipeline_aborted:
    header('☁️ 最終 Drive 同步')
    n = sync_to_drive()
    total_synced += n
    log('✅', f'同步完成: {n} 個新檔案 (總計 {total_synced})')

# ═══════════════════════════════════════════════════════════════
# 完成統計
# ═══════════════════════════════════════════════════════════════
elapsed_sec = time.time() - pipeline_start_time
elapsed_min = round(elapsed_sec / 60, 1)

header('🏁 執行完畢' if not pipeline_aborted else '⛔ 管線已中止')

if pipeline_aborted:
    log('⛔', '因帳號權限不足而中止：')
    log('  ', '1. TaiwanOptionTick / TaiwanFuturesTick 需 backer 或 sponsor 等級')
    log('  ', '2. 請在表單頂部填入有效的 FINMIND_API_TOKEN')
    log('  ', '3. 填入 Token 後重新執行此儲存格即可')
    if days_done > 0:
        log('💾', f'中止前已完成 {days_done} 天，資料已保存')
else:
    actual_rate = round(api_call_count / max(elapsed_sec / 3600, 0.001), 0)
    log('📊', f'═══ 執行統計 ═══')
    log('  ', f'處理天數:  {days_done}/{total_days} 天')
    log('  ', f'下載成功:  {days_downloaded} 天')
    if not SKIP_GREEKS:
        log('  ', f'Greeks:   {days_greeks} 天')
    log('  ', f'無資料:    {days_skipped} 天')
    log('  ', f'API 呼叫:  {api_call_count} 次')
    log('  ', f'實際速率:  {actual_rate:.0f} 次/hr (額度: {API_QUOTA_PER_HOUR}/hr)')
    log('  ', f'Drive 同步: {total_synced} 個檔案')
    log('  ', f'總耗時:    {elapsed_min} 分鐘')
    log('🎉', '管線執行完畢！下次執行會自動跳過已完成的任務。')
